# Hosted Code Execution

This repo already teaches agents that **write** code — `05_SWE_Agent_Applied.ipynb` and
`3. AI_Agents_with_LangGraph/05_Reflective_Code_Generation_Agent/`. Neither runs it. The
code comes back as text and a human decides what to do with it.

Here the model writes Python **and executes it in a sandbox on the provider's
infrastructure**, then answers using the real result. You maintain no runtime.

There is a second reason this notebook exists. `07_Hosted_vs_Client_Side_Tools.ipynb`
concludes that you cannot see a hosted tool's raw result. That is true of `web_search`.
It is **false** for code interpreter, which hands back the exact source it ran. Hosted
tools are not uniformly opaque, and that distinction is worth having.

## Learning objectives

1. Say why executing code beats predicting an answer, and show it on a task that proves it.
2. Call the hosted `code_interpreter` tool and read back **the code it ran**.
3. Configure the sandbox — memory ceiling and network policy — so "hosted" is not a
   synonym for "uncontrolled".
4. Place this against a client-side sandbox and say which you would run.

## Where this fits

- `07_Hosted_vs_Client_Side_Tools.ipynb` — the general trade-off. Read it first; this
  notebook amends one of its conclusions.
- `05_SWE_Agent_Applied.ipynb` — a code agent whose execution you own.

## Prerequisites and cost

`OPENAI_API_KEY`, on an account with code interpreter enabled. **This tool bills per
container session on top of tokens**, which is a different cost shape from per-call tools —
a long session with many executions is one container, a burst of separate requests is many.

In [ ]:
# ============ SETUP ============
import json
import textwrap

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

MODEL = "gpt-4o-mini"

# Verified against the installed SDK (openai.types.responses.tool_param):
#   type      -> Literal["code_interpreter"]
#   container -> a container id, OR {"type": "auto", file_ids?, memory_limit?, network_policy?}
# Note the literal is "auto" — the standalone SDK class is named ContainerAuto, which is a
# different thing from the value the tool param expects.
CODE_TOOL = {"type": "code_interpreter", "container": {"type": "auto"}}

## 1. A task where predicting the answer fails

The point of executing code is not convenience, it is **determinism**. A model asked to do
real arithmetic over many values is sampling plausible digits; the same model writing three
lines of Python and running them is computing.

The task below is deliberately tedious rather than clever — there is nothing to reason
about, only bookkeeping to get exactly right, which is precisely where token prediction
degrades.

In [ ]:
# ============ THE TASK ============
READINGS = [
    12.4, 9.8, 14.1, 22.7, 8.3, 19.9, 31.2, 7.6, 15.5, 28.0,
    11.1, 24.6, 6.9, 17.8, 13.2, 20.4, 9.1, 26.3, 18.7, 10.5,
]

TASK = (
    "Here are 20 sensor readings:\n"
    f"{READINGS}\n\n"
    "Report: the mean to 4 decimal places, the sample standard deviation to 4 decimal "
    "places, and how many readings lie more than one sample standard deviation above the "
    "mean. Give the three numbers plainly."
)

# ground truth, computed locally, so we can grade the model rather than trust it
import statistics
TRUE_MEAN = statistics.mean(READINGS)
TRUE_SD = statistics.stdev(READINGS)
TRUE_ABOVE = sum(1 for r in READINGS if r > TRUE_MEAN + TRUE_SD)
print(f"  ground truth: mean={TRUE_MEAN:.4f}  sd={TRUE_SD:.4f}  above={TRUE_ABOVE}")

In [ ]:
# ============ WITHOUT EXECUTION: THE MODEL GUESSES ============
guessed = client.responses.create(model=MODEL, input=TASK)
print(guessed.output_text)
print(f"\n  (ground truth: {TRUE_MEAN:.4f} / {TRUE_SD:.4f} / {TRUE_ABOVE})")

### Discussion of the output

Compare digit by digit. The mean is often close — means are forgiving. The standard
deviation is usually wrong past the second decimal, and the count above one SD is a coin
toss, because it requires holding the other two results and re-scanning twenty values.

The failure mode matters more than the error size: the answer arrives **formatted
confidently to four decimal places**. Nothing in the reply signals that the last two digits
were invented.

## 2. With hosted execution

The same prompt plus one tool. No function of yours, no sandbox to run, no result to feed
back — the model writes Python, runs it on OpenAI's side, and answers from the output.

In [ ]:
# ============ WITH THE HOSTED SANDBOX ============
executed = client.responses.create(model=MODEL, input=TASK, tools=[CODE_TOOL])
print(executed.output_text)
print(f"\n  (ground truth: {TRUE_MEAN:.4f} / {TRUE_SD:.4f} / {TRUE_ABOVE})")

## 3. The part `07_` said you would not get

Read the source it executed. This is the distinction that notebook does not draw: a hosted
`web_search` returns a conclusion, while a hosted `code_interpreter` returns **its working**.

In [ ]:
# ============ READ BACK THE CODE IT RAN ============
calls = [i for i in executed.output if getattr(i, "type", "") == "code_interpreter_call"]
print(f"  code_interpreter calls: {len(calls)}\n")

for n, call in enumerate(calls, 1):
    print(f"  --- call {n} · status={call.status} · container={call.container_id}")
    if call.code:
        print(textwrap.indent(call.code.strip(), "      "))
    for out in (call.outputs or []):
        kind = getattr(out, "type", "?")
        if kind == "logs":
            print(textwrap.indent(f"[stdout] {out.logs.strip()}", "      "))
        elif kind == "image":
            print(f"      [image] {out.url}")
    print()

### Why that changes the picture

You can now audit the computation. Not the *result* — the **method**. If the number is
wrong you can see whether it used the population or sample standard deviation, whether the
comparison was strict or inclusive, whether it silently dropped a value.

That is a materially different debugging position from `web_search`, where a wrong answer
leaves you nothing to inspect. Revise the rule from `07_` to:

> **How much of a hosted tool you can see is a property of that tool, not of hosted tools.**

| Hosted tool | What comes back |
|---|---|
| `web_search` | a conclusion, and roughly what was searched |
| `code_interpreter` | the **source it ran**, stdout, and any files produced |
| `computer_use_preview` | the actions it took |

## 4. Hosted does not mean unconfigured

The second half of `07_`'s argument was that hosted tools remove the seam where you enforce
policy. Code interpreter keeps part of it — the container is configurable, and the two knobs
that matter are a **memory ceiling** and a **network policy**.

Both shapes below are verified against the installed SDK
(`ContainerNetworkPolicyDisabledParam`, `ContainerNetworkPolicyAllowlistParam`).

In [ ]:
# ============ A HARDENED CONTAINER ============
# network_policy "disabled" is the one to reach for by default: a sandbox that computes on
# data you supplied has no business making outbound requests.
HARDENED = {
    "type": "code_interpreter",
    "container": {
        "type": "auto",
        "memory_limit": "1g",                    # 1g | 4g | 16g | 64g
        "network_policy": {"type": "disabled"},  # or {"type":"allowlist","allowed_domains":[...]}
    },
}

hardened = client.responses.create(model=MODEL, input=TASK, tools=[HARDENED])
print(hardened.output_text[:300])

calls = [i for i in hardened.output if getattr(i, "type", "") == "code_interpreter_call"]
print(f"\n  ran in a 1g, network-disabled container · calls={len(calls)}")

### The honest comparison with client-side

Configurable is not the same as controllable. What you still cannot do:

- **See the code before it runs.** You get it back after execution, not for approval first.
- **Veto a specific call.** There is no seam between decision and execution.
- **Reproduce it in a test.** A client-side sandbox can be stubbed; this cannot.

What you no longer have to do: build a container, pin its packages, sandbox it, bound its
memory, cut its network, and keep all of that patched. `05_SWE_Agent_Applied.ipynb` is the
version where that work is yours.

So the trade is narrower here than for `web_search` but the same in shape: **you exchange
pre-execution control for not operating a runtime.**

## 5. Choosing

| Reach for **hosted execution** when | Run your **own** sandbox when |
|---|---|
| The code is disposable — analysis, a chart, a conversion | The code touches your systems, secrets or customer data |
| You want determinism without operating a runtime | You must approve code **before** it executes |
| Sessions are short and bounded | You need reproducible tests, stubbed execution, or an audit trail you own |
| A memory cap and no network are sufficient policy | Policy means package pinning, egress rules, or tenant isolation |

The default worth holding: **hosted for arithmetic and analysis on data you passed in;
your own runtime for anything that touches something real.**

## Key takeaways

1. **Executing beats predicting.** A model computing twenty values in Python is exact; the
   same model answering directly is sampling digits — and formats them just as confidently.
2. **Code interpreter returns the source it ran.** You can audit the *method*, which is a
   far better debugging position than a bare conclusion.
3. **So `07_`'s "you cannot see a hosted tool's result" is too general.** Visibility is a
   property of the specific tool.
4. **The container is configurable** — `memory_limit` and `network_policy`. Default to
   `{"type": "disabled"}`: a sandbox working on data you supplied should not reach the network.
5. **You still cannot approve code before it runs**, veto a call, or stub it in a test. That
   is the remaining gap versus a sandbox you operate.
6. **It bills per container session**, not per call — a different cost shape from the other
   hosted tools, and one that rewards batching work into one session.

### Next

- `05_SWE_Agent_Applied.ipynb` — the same capability with the runtime, and the operating
  cost, on your side.